<a href="https://colab.research.google.com/github/yy726/tool_use_demo/blob/main/notebooks/03_demo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 03 - Demo: try the planner

Serves the trained checkpoints behind OpenAI-compatible endpoints and opens a small web app where reviewers can run the
five preset hard queries, or their own, and compare checkpoints side by side.

How the pieces fit:

- **Model server.** `SERVER = "unsloth"` runs Unsloth Studio's `unsloth run` on the GGUF builds pushed by the training
  notebooks. `SERVER = "vllm"` serves the merged weights with vLLM instead. `SERVER = "external"` uses endpoints you
  already run somewhere (for example Unsloth Desktop on your own machine).
- **Tool loop.** Unsloth's chat window can only run its own built-in tools, not our four. So this notebook's client
  sends our tool schemas with each request, executes the calls the model asks for against the local mock world and
  knowledge base, and returns the results - at most three rounds.

A T4 is enough for `SERVER = "unsloth"`; `SERVER = "vllm"` needs an L4 or A100.

In [ ]:
HF_USER = ""                 # your Hugging Face user name ("" = take it from the HF_TOKEN secret)
MODEL_KEY = "qwen3-1.7b"     # must match the training notebooks
SMOKE = False                # True to serve the "-smoke" repositories from a SMOKE training run
STAGES = ["sft", "grpo"]     # trained checkpoints to serve
INCLUDE_BASE = True          # also serve the untuned base model for comparison
SERVER = "unsloth"           # "unsloth" | "vllm" | "external"
EXTERNAL = {}                # for SERVER = "external": {"grpo": ("https://host/v1", "sk-unsloth-...")}
GGUF_VARIANT = "Q8_0"
SHARE = True                 # public Gradio link for reviewers (valid while this runtime is alive)

## 1. Install and set up

In [ ]:
!pip install -q openai gradio bm25s==0.3.16 "datasets>=4.7.0,<5.0.0" huggingface_hub
if SERVER == "unsloth":
    !curl -fsSL https://unsloth.ai/install.sh | sh
elif SERVER == "vllm":
    !pip install -q vllm

In [ ]:
import os, subprocess, sys

REPO_URL = "https://github.com/yy726/tool_use_demo.git"
REPO_BRANCH = "main"

if os.path.isdir("planner"):                      # already inside a checkout
    REPO_DIR = os.getcwd()
elif os.path.isdir("../planner"):
    REPO_DIR = os.path.abspath("..")
else:
    REPO_DIR = "/content/tool_use_demo"
    if not os.path.isdir(REPO_DIR):
        url = REPO_URL
        try:                                      # private repo: add a GITHUB_TOKEN secret
            from google.colab import userdata
            token = userdata.get("GITHUB_TOKEN")
            if token: url = url.replace("https://", f"https://{token}@")
        except Exception:
            pass
        subprocess.run(["git", "clone", "--depth", "1", "--branch", REPO_BRANCH, url, REPO_DIR], check=True)
    else:                                         # an older clone: move it to the latest commit of REPO_BRANCH
        for cmd in (["fetch", "--depth", "1", "origin", REPO_BRANCH], ["checkout", "-f", "-B", REPO_BRANCH, "FETCH_HEAD"]):
            done = subprocess.run(["git", "-C", REPO_DIR] + cmd, capture_output=True, text=True)
            if done.returncode: print("WARNING: git", cmd[0], "failed, the code in", REPO_DIR, "may be stale:", done.stderr.strip())
os.chdir(REPO_DIR)
if REPO_DIR not in sys.path: sys.path.insert(0, REPO_DIR)
head = subprocess.run(["git", "-C", REPO_DIR, "log", "-1", "--format=%h %s"], capture_output=True, text=True).stdout.strip()
print("repo:", REPO_DIR, "@", head)

In [ ]:
from planner.config import get_config
from planner.persist import read_secrets
from planner.setup_env import setup_environment

cfg = get_config("smoke" if SMOKE else "a100", MODEL_KEY)
read_secrets(("HF_TOKEN",))
if not HF_USER and os.environ.get("HF_TOKEN"):
    from huggingface_hub import whoami
    HF_USER = whoami(token=os.environ["HF_TOKEN"])["name"]
assert HF_USER or SERVER == "external", "Set HF_USER (or add an HF_TOKEN secret)."

# Rebuild the same world and knowledge base the models were trained against (about a minute).
world, corpus = setup_environment(cache_dir="/content/planner_env", n_train=cfg.hotpot_train_rows, n_val=cfg.hotpot_val_rows, seed=cfg.seed)

## 2. Start the model servers

Each checkpoint gets its own port. With `unsloth run`, the API key it prints on start-up is read from its log.

In [ ]:
import shutil
from planner.demo import OpenAIBackend
from planner.serve import Endpoint, start_unsloth, start_vllm, wait_ready

os.environ["PATH"] = os.path.expanduser("~/.local/bin") + os.pathsep + os.path.expanduser("~/.unsloth/bin") + os.pathsep + os.environ["PATH"]
names = (["base"] if INCLUDE_BASE else []) + STAGES
endpoints = {}
if SERVER == "external":
    endpoints = {name: Endpoint(name, url, key) for name, (url, key) in EXTERNAL.items()}
else:
    for i, name in enumerate(names):
        port, log = 8001 + i, f"/content/server_{name}.log"
        try:
            if SERVER == "unsloth":
                assert shutil.which("unsloth"), "the `unsloth` command is not on PATH: check the install cell's output"
                repo = f"unsloth/{cfg.base_model_hf.split('/')[-1]}-GGUF" if name == "base" else cfg.repo(HF_USER, name, "GGUF")
                endpoints[name] = wait_ready(start_unsloth(name, repo, port, log, GGUF_VARIANT), key_from_log=True)
            else:
                repo = cfg.base_model_hf if name == "base" else cfg.repo(HF_USER, name)
                endpoints[name] = wait_ready(start_vllm(name, repo, port, log, gpu_memory_utilization=round(0.85 / len(names), 2)))
            print(f"{name}: ready at {endpoints[name].base_url}")
        except Exception as error:
            print(f"{name}: could not start ({error})")

backends = {name: OpenAIBackend(e.base_url, e.api_key) for name, e in endpoints.items()}
assert backends, "No endpoint is up. Check /content/server_*.log, or try SERVER = 'vllm'."
print({name: b.model for name, b in backends.items()})

## 3. Five hard queries

| # | type | what a good plan does |
|---|---|---|
| 1 | bridge / multi-hop | search, then search again with an entity found in the first result |
| 2 | decomposition + fan-out + synthesis | look up two entities in parallel, then compare |
| 3 | multi-date fan-out + calculation | three weather lookups in one round, then the calculator |
| 4 | cross-tool bridge + tool selection | `get_election` gives the capital and the date for `get_weather` |
| 5 | multi-part ask + query reformulation | split the question; the knowledge part needs a follow-up query |

They come from the held-out set, so each has a reference plan and gold answer, and the output shows the relevance and
factuality scores along with failure tags.

In [ ]:
from IPython.display import Markdown, display
from planner.demo import answer, format_expected, format_markdown, load_examples

examples = load_examples()
for i, example in enumerate(examples, 1):
    task = example["task"]
    display(Markdown(f"## {i}. {example['label']}\n\n> {task['query']}\n\n{example['why']}\n\n{format_expected(task)}"))
    for name, backend in backends.items():
        try:
            display(Markdown(format_markdown(answer(task, backend), title=name)))
        except Exception as error:
            print(f"{name}: request failed ({error})")

## 4. Web app for reviewers

Pick a preset or type any question. With `SHARE = True` Gradio prints a public `gradio.live` link that works while this
runtime is running. Weather and elections belong to a fictional world (regions such as the ones in the presets), so
ask about those; `knowledge_search` covers the Wikipedia paragraphs of the HotpotQA questions used to build the index.

In [ ]:
from planner.demo import build_ui

ui = build_ui(backends, examples)
ui.launch(share=SHARE, debug=False)

## Hosting it yourself with Unsloth Desktop / Studio

On your own machine, with [Unsloth](https://unsloth.ai/docs/get-started/unsloth-studio) installed:

```bash
unsloth run --model <hf-user>/qwen3-1.7b-planner-grpo-GGUF:Q8_0 -p 8888 --disable-tools
```

It prints the endpoint URL and an API key. Then, from a clone of this repository:

```bash
python -m planner.demo --base-url http://127.0.0.1:8888/v1 --api-key sk-unsloth-... --example 4
```

`unsloth studio --secure` publishes the endpoint on a Cloudflare HTTPS URL for remote reviewers; use that URL as
`--base-url` (the client already sends `stream: false`, which the tunnel requires). You can also load the GGUF in the
Unsloth chat window to talk to the model directly, but its tool calls will not be executed there: only the client in
this repository runs the four planner tools.

When you are done, stop the servers:

In [ ]:
for endpoint in endpoints.values(): endpoint.stop()